# PyTorch GPU and Performance Tutorial

This tutorial covers GPU acceleration and performance optimization for
kgcnn-torch models using native PyTorch features:

1. **Device selection** with `kgcnn_torch.utils.devices`
2. **Model and data placement** on GPU with `.to(device)`
3. **`torch.compile()`** for model optimization (PyTorch 2.0+)
4. **Mixed precision training** with `torch.amp`
5. **DataLoader parallelism** with `num_workers`
6. **Memory management**: `empty_cache()`, gradient checkpointing

This replaces the Keras `tutorial_jax_jit.ipynb` notebook, since JAX JIT
compilation is not relevant for the PyTorch backend.

## 1. Device Selection

kgcnn-torch provides device utilities in `kgcnn_torch.utils.devices`.
These functions detect available GPUs and help select the appropriate device.

In [ ]:
import torch
from kgcnn_torch.utils.devices import check_device, get_device, set_cuda_device, get_gpu_memory_info

# Check what devices are available
device_info = check_device()
for key, value in device_info.items():
    print(f"{key}: {value}")

In [ ]:
# get_device() automatically selects CUDA if available, otherwise CPU
device = get_device("auto")
print(f"Selected device: {device}")

# You can also explicitly choose a device:
# device = get_device("cpu")       # Force CPU
# device = get_device("cuda:0")    # Specific GPU
# device = get_device("cuda:1")    # Second GPU

In [ ]:
# For multi-GPU systems, you can set the default CUDA device:
# set_cuda_device(0)  # Use GPU 0

# Or via environment variable (recommended, set BEFORE importing torch):
# import os
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Only expose GPU 0

# Check GPU memory (if available)
if torch.cuda.is_available():
    mem = get_gpu_memory_info(0)
    print(f"GPU 0 memory:")
    print(f"  Total:     {mem['total'] / 1e9:.2f} GB")
    print(f"  Allocated: {mem['allocated'] / 1e9:.2f} GB")
    print(f"  Cached:    {mem['cached'] / 1e9:.2f} GB")
    print(f"  Free:      {mem['free'] / 1e9:.2f} GB")
else:
    print("No CUDA GPU available, running on CPU.")

## 2. Model and Data on GPU

In PyTorch, you must explicitly move models and data to the GPU.
Both the model and input data must be on the same device.

In [ ]:
import numpy as np
from torch_geometric.loader import DataLoader
from kgcnn_torch.models.gin import GINModel
from kgcnn_torch.data.datasets.FreeSolvDataset import FreeSolvDataset

# Load dataset
dataset = FreeSolvDataset()
print(f"FreeSolv: {len(dataset)} graphs")

In [ ]:
# Create model and move to device
model = GINModel(
    node_dim=64,
    depth=3,
    units=64,
    gin_mlp_units=[64, 64],
    gin_mlp_activation="relu",
    gin_mlp_use_normalization=False,
    use_edge_features=True,
    edge_dim=11,
    node_pooling="sum",
    output_units=[],
    output_final_activation="linear",
    num_targets=1,
    use_node_embedding=True,
    num_embeddings=95,
)

# Move model to GPU (or CPU)
model = model.to(device)
print(f"Model on device: {next(model.parameters()).device}")

In [ ]:
# The DataLoader returns batches on CPU by default.
# Move each batch to the device in the training loop.
from sklearn.model_selection import train_test_split

indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_loader = DataLoader(dataset[torch.tensor(train_idx).long()], batch_size=32, shuffle=True)
test_loader = DataLoader(dataset[torch.tensor(test_idx).long()], batch_size=32)

# Example: moving data to device
batch = next(iter(train_loader))
print(f"Batch before .to(device): {batch.x.device if hasattr(batch, 'x') and batch.x is not None else 'N/A'}")

batch = batch.to(device)
print(f"Batch after .to(device):  {batch.edge_index.device}")
print(f"Batch info: {batch}")

The `kgcnn_torch.training.trainer.fit()` function handles `.to(device)` automatically
when you pass the `device` parameter:

In [ ]:
import torch.nn as nn
from kgcnn_torch.training.trainer import fit

# Training with automatic device handling
history = fit(
    model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    loss_fn=nn.L1Loss(),
    epochs=50,
    device=device,   # <-- handles batch.to(device) internally
    verbose=1,
)

print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss:   {history['val_loss'][-1]:.4f}")

## 3. torch.compile() for Model Optimization

PyTorch 2.0+ introduced `torch.compile()`, which JIT-compiles the model
using TorchDynamo and TorchInductor for faster execution.

This is the PyTorch equivalent of JAX's `jit` compilation.

In [ ]:
import time

# Create a fresh model for benchmarking
model_eager = GINModel(
    node_dim=64, depth=3, units=64, gin_mlp_units=[64, 64],
    gin_mlp_activation="relu", gin_mlp_use_normalization=False,
    use_edge_features=True, edge_dim=11, node_pooling="sum",
    output_units=[], output_final_activation="linear",
    num_targets=1, use_node_embedding=True, num_embeddings=95,
).to(device)

# Compile the model (PyTorch 2.0+)
# mode="default" gives a good balance of compile time and speedup
# mode="reduce-overhead" minimizes framework overhead (best for small models)
# mode="max-autotune" takes longer to compile but is fastest at runtime
try:
    model_compiled = torch.compile(model_eager, mode="default")
    print("Model compiled with torch.compile()")
    print(f"PyTorch version: {torch.__version__}")

    # Warmup pass (compilation happens on first call)
    print("\nRunning warmup (compilation pass)...")
    batch = next(iter(train_loader)).to(device)
    with torch.no_grad():
        _ = model_compiled(batch)
    print("Warmup complete.")

    # Benchmark compiled vs eager
    model_eager.eval()
    model_compiled.eval()
    n_runs = 100

    # Eager timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model_eager(batch)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    eager_time = (time.perf_counter() - t0) / n_runs

    # Compiled timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model_compiled(batch)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    compiled_time = (time.perf_counter() - t0) / n_runs

    print(f"\nEager mode:    {eager_time*1000:.2f} ms/batch")
    print(f"Compiled mode: {compiled_time*1000:.2f} ms/batch")
    print(f"Speedup:       {eager_time/compiled_time:.2f}x")

except Exception as e:
    print(f"torch.compile() not available or failed: {e}")
    print("This requires PyTorch >= 2.0")

## 4. Mixed Precision Training

Mixed precision (FP16/BF16) uses lower-precision arithmetic for most operations
while keeping critical accumulations in FP32. This can:
- **Reduce memory** by ~2x (FP16 uses half the memory of FP32)
- **Speed up computation** on GPUs with Tensor Cores (Volta, Ampere, Hopper)

PyTorch provides `torch.amp` (Automatic Mixed Precision) for this.

In [ ]:
from torch.amp import autocast, GradScaler

# Create a fresh model
model_amp = GINModel(
    node_dim=64, depth=3, units=64, gin_mlp_units=[64, 64],
    gin_mlp_activation="relu", gin_mlp_use_normalization=False,
    use_edge_features=True, edge_dim=11, node_pooling="sum",
    output_units=[], output_final_activation="linear",
    num_targets=1, use_node_embedding=True, num_embeddings=95,
).to(device)

optimizer = torch.optim.Adam(model_amp.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()

# GradScaler prevents underflow in FP16 gradients
use_amp = torch.cuda.is_available()  # AMP requires CUDA
scaler = GradScaler(enabled=use_amp)

# AMP device type: "cuda" for NVIDIA GPUs
amp_device_type = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Mixed precision enabled: {use_amp}")
print(f"AMP device type: {amp_device_type}")

In [ ]:
# Manual training loop with AMP
model_amp.train()
epoch_losses = []

for epoch in range(20):
    running_loss = 0.0
    n_batches = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Forward pass under autocast (uses FP16 where safe)
        with autocast(device_type=amp_device_type, enabled=use_amp):
            pred = model_amp(batch)
            target = batch.y.unsqueeze(-1) if batch.y.dim() == 1 else batch.y
            loss = loss_fn(pred, target)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        n_batches += 1

    avg_loss = running_loss / n_batches
    epoch_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}: loss = {avg_loss:.4f}")

print(f"\nFinal loss: {epoch_losses[-1]:.4f}")

## 5. DataLoader with `num_workers`

PyTorch's DataLoader supports parallel data loading using multiple worker
processes. This overlaps data loading with GPU computation.

Key parameters:
- `num_workers`: Number of subprocesses for data loading. 0 means data is loaded in the main process.
- `pin_memory`: If True, copies tensors to CUDA pinned memory (faster GPU transfers).
- `prefetch_factor`: Number of batches to prefetch per worker.

In [ ]:
# Standard DataLoader (single-process)
loader_single = DataLoader(
    dataset[torch.tensor(train_idx).long()],
    batch_size=32,
    shuffle=True,
    num_workers=0,     # Data loading in main process
)

# Optimized DataLoader with parallel loading
loader_parallel = DataLoader(
    dataset[torch.tensor(train_idx).long()],
    batch_size=32,
    shuffle=True,
    num_workers=2,          # 2 worker processes (adjust based on CPU cores)
    pin_memory=True,        # Faster CPU->GPU transfer
    persistent_workers=True, # Keep workers alive between epochs
    prefetch_factor=2,      # Prefetch 2 batches per worker
)

print(f"Single-process loader: {len(loader_single)} batches")
print(f"Multi-process loader:  {len(loader_parallel)} batches")

In [ ]:
# Benchmark: single vs parallel data loading
model_bench = GINModel(
    node_dim=64, depth=3, units=64, gin_mlp_units=[64, 64],
    gin_mlp_activation="relu", gin_mlp_use_normalization=False,
    use_edge_features=True, edge_dim=11, node_pooling="sum",
    output_units=[], output_final_activation="linear",
    num_targets=1, use_node_embedding=True, num_embeddings=95,
).to(device)
model_bench.eval()

def time_loader(loader, model, device, n_epochs=3):
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_epochs):
            for batch in loader:
                batch = batch.to(device)
                _ = model(batch)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.perf_counter() - t0

t_single = time_loader(loader_single, model_bench, device)
t_parallel = time_loader(loader_parallel, model_bench, device)

print(f"Single-process: {t_single:.2f}s")
print(f"Multi-process:  {t_parallel:.2f}s")
print(f"Speedup:        {t_single/t_parallel:.2f}x")

## 6. Memory Management

GPU memory is a precious resource. Here are techniques to manage it effectively.

### 6.1 Monitoring GPU Memory

In [ ]:
def print_gpu_memory(label=""):
    """Print current GPU memory usage."""
    if not torch.cuda.is_available():
        print(f"[{label}] No CUDA GPU available.")
        return
    allocated = torch.cuda.memory_allocated() / 1e6
    cached = torch.cuda.memory_reserved() / 1e6
    print(f"[{label}] Allocated: {allocated:.1f} MB, Cached: {cached:.1f} MB")

print_gpu_memory("Before model creation")

# Create a large model
large_model = GINModel(
    node_dim=128, depth=6, units=128, gin_mlp_units=[128, 128],
    gin_mlp_activation="relu", gin_mlp_use_normalization=False,
    use_edge_features=True, edge_dim=11, node_pooling="sum",
    output_units=[128, 64], output_final_activation="linear",
    num_targets=1, use_node_embedding=True, num_embeddings=95,
).to(device)
print_gpu_memory("After model creation")

### 6.2 Clearing GPU Cache

`torch.cuda.empty_cache()` releases unused cached memory back to the OS.
Useful after deleting models or between experiments.

In [ ]:
print_gpu_memory("Before cleanup")

# Delete the model
del large_model
print_gpu_memory("After del model (cache still held)")

# Release cached memory back to the OS
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print_gpu_memory("After empty_cache()")

### 6.3 Gradient Checkpointing

For very large GNNs, gradient checkpointing trades computation for memory.
Instead of storing all intermediate activations during the forward pass,
it recomputes them during the backward pass.

Use `torch.utils.checkpoint.checkpoint()` to wrap expensive submodules.

In [ ]:
from torch.utils.checkpoint import checkpoint

class GINModelWithCheckpointing(GINModel):
    """GIN model variant that uses gradient checkpointing for GIN layers.

    Reduces memory usage at the cost of recomputing activations during backward.
    Useful for very deep networks or large graphs.
    """

    def __init__(self, use_checkpointing=True, **kwargs):
        super().__init__(**kwargs)
        self.use_checkpointing = use_checkpointing

    def forward(self, data):
        edge_index = data.edge_index
        batch = data.batch

        # Edge features (for GINE)
        edge_attr = None
        if self.use_edge_features:
            edge_attr = data.edge_attr
            if self.edge_proj is not None:
                edge_attr = self.edge_proj(edge_attr)

        # Node embedding
        if self.use_node_embedding:
            x = data.z if hasattr(data, 'z') and data.z is not None else data.x
            x = self.node_embedding(x.long())
        else:
            x = data.x
            x = self.node_projection(x)

        x = self.dense_in(x)
        list_embeddings = [x]

        for i in range(self.depth):
            # Use gradient checkpointing for each GIN layer
            if self.use_checkpointing and self.training:
                def create_layer_fn(layer_idx):
                    def layer_fn(node_feat):
                        if self.use_edge_features:
                            h = self.convs[layer_idx](node_feat, edge_index, edge_attr)
                        else:
                            h = self.convs[layer_idx](node_feat, edge_index)
                        h = self.gin_mlps[layer_idx](h)
                        return h
                    return layer_fn
                x = checkpoint(create_layer_fn(i), x, use_reentrant=False)
            else:
                if self.use_edge_features:
                    x = self.convs[i](x, edge_index, edge_attr)
                else:
                    x = self.convs[i](x, edge_index)
                x = self.gin_mlps[i](x)
            list_embeddings.append(x)

        # Readout (same as parent)
        batch_size = int(batch.max().item()) + 1 if batch.numel() > 0 else 1
        out = None
        for i, emb in enumerate(list_embeddings):
            h = self.pooling(emb, batch, batch_size)
            h = self.readout_mlps[i](h)
            h = self.readout_dropouts[i](h)
            out = h if out is None else out + h
        out = self.output_mlp(out)
        return out


# Usage
model_ckpt = GINModelWithCheckpointing(
    use_checkpointing=True,
    node_dim=64, depth=3, units=64, gin_mlp_units=[64, 64],
    gin_mlp_activation="relu", gin_mlp_use_normalization=False,
    use_edge_features=True, edge_dim=11, node_pooling="sum",
    output_units=[], output_final_activation="linear",
    num_targets=1, use_node_embedding=True, num_embeddings=95,
).to(device)
print("Gradient checkpointing model created.")

# Train briefly to verify it works
opt = torch.optim.Adam(model_ckpt.parameters(), lr=1e-3)
model_ckpt.train()
batch = next(iter(train_loader)).to(device)
opt.zero_grad()
pred = model_ckpt(batch)
target = batch.y.unsqueeze(-1) if batch.y.dim() == 1 else batch.y
loss = nn.L1Loss()(pred, target)
loss.backward()
opt.step()
print(f"Forward + backward pass successful. Loss: {loss.item():.4f}")

## Summary of Performance Techniques

| Technique | When to Use | Expected Benefit |
|---|---|---|
| `model.to(device)` | Always for GPU training | Required for GPU usage |
| `torch.compile()` | PyTorch 2.0+, stable models | 1.2-2x speedup |
| Mixed Precision (`torch.amp`) | NVIDIA GPUs with Tensor Cores | 1.5-3x speedup, 50% less memory |
| `num_workers > 0` | Large datasets, I/O-bound training | Overlaps data loading with computation |
| `pin_memory=True` | CUDA training | Faster CPU-to-GPU transfers |
| `torch.cuda.empty_cache()` | Between experiments | Frees cached GPU memory |
| Gradient Checkpointing | Large models / deep GNNs | Trades ~30% speed for ~50% memory savings |